## Kalshi Raw Data - API

In [1]:
import requests
import pandas as pd
import time

In [ ]:
# Defining a function to extract historical trades data from Kalshi API

def get_all_historical_trades(ticker):
    base_url = "https://external-api.kalshi.com/trade-api/v2/historical/trades"
    all_trades = []
    cursor = None

    headers = {
        "accept": "application/json"
    }
    
    print(f"Starting data extraction for: {ticker}...")
    
    while True:
        params = {
            "ticker": ticker,
            "limit": 1000  
        }
        
        if cursor:
            params["cursor"] = cursor
            
        response = requests.get(base_url, headers=headers, params=params)

        if response.status_code != 200:
            print(f"API Error {response.status_code}: {response.text}")
            break
            
        data = response.json()
        trades = data.get("trades", [])
        all_trades.extend(trades)

        cursor = data.get("cursor")
        if not cursor:
            break

        time.sleep(0.2) # Pause between pages to avoid overloading server

    print(f"Extraction complete fetched {len(all_trades)} individual trades")

    df = pd.DataFrame(all_trades)
    return df

In [3]:
# Change the target ticker as per your requirements 
target_ticker = "KXFEDDECISION-26MAR-C25"
historic_df = get_all_historical_trades(target_ticker)

historic_df["created_time"] = pd.to_datetime(historic_df["created_time"])
historic_df = historic_df.rename(columns={"count_fp":"count"})
historic_df = historic_df.sort_values("created_time").reset_index(drop = True)

desired_order = [
    "created_time",
    "count",
    "no_price_dollars",
    "taker_side",
    "ticker",
    "trade_id",
    "yes_price_dollars",
]

historic_df = historic_df[desired_order]

Starting data extraction for: KXFEDDECISION-26MAR-C25...
Extraction complete! Successfully fetched 18463 individual trades.


In [ ]:
# Viewing the dataframe 
historic_df

,created_time,count,no_price_dollars,taker_side,ticker,trade_id,yes_price_dollars
0,2025-10-30 00:57:57.264521+00:00,100.00,0.5800,yes,KXFEDDECISION-26MAR-C25,54d001a5-30ef-695f-8a6a-6ae06d31efb4,0.4200
1,2025-11-05 20:28:50.250022+00:00,99.00,0.6300,no,KXFEDDECISION-26MAR-C25,c7492216-a1e0-695b-965e-2b3fc95ebf0b,0.3700
2,2025-11-06 06:21:43.725681+00:00,1.00,0.6400,no,KXFEDDECISION-26MAR-C25,0aa06488-8a39-42c8-23e8-d9769e926405,0.3600
3,2025-11-12 09:18:34.025787+00:00,15.00,0.6400,no,KXFEDDECISION-26MAR-C25,54e87c6d-1d65-7021-6939-601f55c7f8d8,0.3600
4,2025-12-10 19:12:12.860892+00:00,15.00,0.6900,yes,KXFEDDECISION-26MAR-C25,2adc9d4c-58c0-5e01-fb21-6d87508fcb90,0.3100
...,...,...,...,...,...,...,...
18458,2026-03-18 17:58:23.087834+00:00,93.00,0.9900,yes,KXFEDDECISION-26MAR-C25,0001e4aa-4790-7f4d-f5aa-a606a2438cd1,0.0100
18459,2026-03-18 17:58:38.202526+00:00,935.00,0.9900,yes,KXFEDDECISION-26MAR-C25,811b465b-6638-5569-9bd5-8781efe5fdb1,0.0100
18460,2026-03-18 17:58:40.789916+00:00,833.00,0.9900,yes,KXFEDDECISION-26MAR-C25,0b47cab8-9804-5b31-8161-7ab1ad16c795,0.0100
18461,2026-03-18 17:58:52.313853+00:00,93.00,0.9900,yes,KXFEDDECISION-26MAR-C25,6be5aac0-e7b4-6751-72d6-6c7ea8326fb1,0.0100


In [ ]:
# Saving the dataframe in the defined directory 
# Ensure that you pick the directory corresponding to the event's data you have extracted in this example for 25bp cut 
historic_df.to_csv("../../../data/raw/Kalshi/data_25bp_cut/kalshi_MAR_2026.csv", index=False)